# SOM (Self-Organizing Map) sur le dataset Réel (Shapes)

Ce notebook applique notre implémentation de SOM (`src/som.py`) sur le jeu de données Shapes (images couleur 3×32×32, soit 3072 caractéristiques, 6 classes de formes géométriques). Même structure que `04_som.ipynb`, avec les adaptations nécessaires pour l'affichage RGB.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.dataset import load_shapes_dataset
from src.helper import extract_full_dataset
from src.som import SOM
from src.metrics import compression_report, Latent

## 1. Chargement des données (Dataset Shapes)

In [3]:
dataloader = load_shapes_dataset(data_dir="../data/shapes_hard_color/train", batch_size=4096, shuffle=True)
images, shape_labels = extract_full_dataset(dataloader)

N = 10000
X = images[:N].flatten(start_dim=1).numpy()
shape_labels = shape_labels[:N].numpy()

class_names = ["bar", "circle", "cross", "square", "star", "triangle"]

print("X shape:", X.shape)
print("Shape labels shape:", shape_labels.shape)

X shape: (10000, 3072)
Shape labels shape: (10000,)


## 2. Choix de la taille de grille

Balayage de plusieurs grilles pour évaluer quantization error et topographic error sur les données Shapes.

In [4]:
GRID_SHAPES = [(4, 4), (8, 8), (10, 10), (16, 16)]

sweep_results = []
for gs in GRID_SHAPES:
    print(f"Training SOM {gs[0]}x{gs[1]} ({gs[0]*gs[1]} neurons)...")
    som = SOM(grid_shape=gs, n_iterations=10000, random_state=42)
    som.fit(X)
    qe = som.quantization_error(X)
    te = som.topographic_error(X)
    sweep_results.append((gs, qe, te))
    print(f"  -> QE = {qe:.4f}, TE = {te:.4f}")

Training SOM 4x4 (16 neurons)...
  -> QE = 8.4794, TE = 0.5384
Training SOM 8x8 (64 neurons)...
  -> QE = 7.8681, TE = 0.5414
Training SOM 10x10 (100 neurons)...
  -> QE = 7.7043, TE = 0.4874
Training SOM 16x16 (256 neurons)...
  -> QE = 7.4023, TE = 0.4779


In [5]:
labels = [f"{gs[0]}x{gs[1]}\n({gs[0]*gs[1]})" for gs, _, _ in sweep_results]
qes = [r[1] for r in sweep_results]
tes = [r[2] for r in sweep_results]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Quantization Error", "Topographic Error"])

fig.add_trace(
    go.Bar(x=labels, y=qes, marker_color="#636efa", name="QE"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=labels, y=tes, marker_color="#EF553B", name="TE"),
    row=1, col=2
)

fig.update_layout(
    title_text="Impact de la taille de grille sur les métriques SOM (Shapes)",
    template="plotly_dark",
    showlegend=False,
    height=400, width=900
)
fig.show()

## 3. Fit avec la grille choisie

On retient une grille 10×10 avec 10000 itérations pour laisser le temps à la SOM de converger sur les 3072 features du dataset couleur.

In [6]:
model = SOM(grid_shape=(10, 10), n_iterations=10000, random_state=42)
model.fit(X)

print(f"Grid: {model.grid_shape}")
print(f"Neurons: {model.n_neurons}")
print(f"Quantization Error: {model.quantization_error(X):.4f}")
print(f"Topographic Error: {model.topographic_error(X):.4f}")

Grid: (10, 10)
Neurons: 100
Quantization Error: 7.7043
Topographic Error: 0.4874


## 4. Courbe de convergence

In [7]:
fig = px.line(
    x=list(range(len(model.history_))),
    y=model.history_,
    labels=dict(x="Échantillon (itération / log_interval)", y="Quantization Error"),
    title="Convergence de la SOM (Dataset Shapes)"
)
fig.update_layout(template="plotly_dark", width=800, height=400)
fig.show()

## 5. Encode / decode / rapport de compression

In [8]:
latent = model.encode(X)
X_reconstructed = model.decode(latent)
codebook = model.get_codebook()

report = compression_report(codebook, latent, X, X_reconstructed)
for k, v in report.items():
    print(f"{k}: {v}")

latent_nature: discrete
codebook_bytes: 1228800
latent_bytes: 10000
total_compressed_bytes: 1238800
original_bytes: 122880000
compression_ratio: 99.19276719405876
reconstruction_mse: 0.020652148872613907


## 6. Visualisation des neurones (codebook)

Chaque neurone possède un vecteur de poids W de dimension 3072. On le reshape en (3, 32, 32) puis on transpose en (32, 32, 3) pour l'afficher comme une image RGB, disposée selon la position du neurone sur la grille.

In [9]:
rows, cols = model.grid_shape

fig = make_subplots(
    rows=rows, cols=cols,
    horizontal_spacing=0.005,
    vertical_spacing=0.005
)

for idx in range(model.n_neurons):
    r = idx // cols + 1
    c = idx % cols + 1
    neuron_img = model.W[idx].reshape(3, 32, 32).transpose(1, 2, 0)
    neuron_img = np.clip(neuron_img, 0.0, 1.0)
    fig.add_trace(
        go.Image(z=(neuron_img * 255).astype(np.uint8)),
        row=r, col=c
    )

for i in range(1, rows * cols + 1):
    fig.update_yaxes(showticklabels=False, row=(i-1)//cols+1, col=(i-1)%cols+1)
    fig.update_xaxes(showticklabels=False, row=(i-1)//cols+1, col=(i-1)%cols+1)

fig.update_layout(
    height=80 * rows, width=80 * cols,
    title_text="Neurones de la SOM (prototypes appris - Shapes)",
    template="plotly_dark"
)
fig.show()

## 7. Visualisation : original vs reconstruit

In [10]:
np.random.seed(42)
indices = np.random.choice(X.shape[0], size=5, replace=False)

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"Original {idx}" for idx in indices] + [f"Reconstructed {idx}" for idx in indices]
)

for i, idx in enumerate(indices):
    orig_img = X[idx].reshape(3, 32, 32).transpose(1, 2, 0)
    recon_img = X_reconstructed[idx].reshape(3, 32, 32).transpose(1, 2, 0)

    orig_img = np.clip(orig_img, 0.0, 1.0)
    recon_img = np.clip(recon_img, 0.0, 1.0)

    fig.add_trace(go.Image(z=(orig_img * 255).astype(np.uint8)), row=1, col=i+1)
    fig.add_trace(go.Image(z=(recon_img * 255).astype(np.uint8)), row=2, col=i+1)

fig.update_layout(
    height=500, width=1000,
    title_text="SOM : Original vs Reconstructed (Shapes)",
    template="plotly_dark"
)

for row in [1, 2]:
    for col in range(1, 6):
        fig.update_yaxes(showticklabels=False, row=row, col=col)
        fig.update_xaxes(showticklabels=False, row=row, col=col)

fig.show()

## 8. Hit Map

In [12]:
bmu_indices = latent.array
hit_counts = np.zeros(model.n_neurons, dtype=int)
for idx in bmu_indices:
    hit_counts[idx] += 1

hit_map = hit_counts.reshape(model.grid_shape)

fig = go.Figure(data=go.Heatmap(
    z=hit_map,
    colorscale="Blues",
    colorbar=dict(title="Nombre de hits"),
    text=hit_map,
    texttemplate="%{text}",
    textfont=dict(size=8)
))
fig.update_layout(
    title="Hit Map (nombre d'échantillons par neurone) - Shapes",
    template="plotly_dark",
    width=600, height=550,
    yaxis=dict(autorange="reversed")
)
fig.show()

n_dead = int(np.sum(hit_counts == 0))
print(f"Neurones morts (0 hits) : {n_dead} / {model.n_neurons}")

Neurones morts (0 hits) : 0 / 100


## 9. Projection 2D colorée par forme

Chaque échantillon est placé à la position de son BMU sur la grille, coloré par sa forme réelle. Si la SOM préserve la topologie, les formes similaires devraient se regrouper.

In [13]:
rng = np.random.default_rng(42)
bmu_coords = model.C[latent.array]
jitter = rng.uniform(-0.3, 0.3, size=bmu_coords.shape)
coords_jittered = bmu_coords + jitter

shape_names = [class_names[lbl] for lbl in shape_labels]

fig = px.scatter(
    x=coords_jittered[:, 1],
    y=coords_jittered[:, 0],
    color=shape_names,
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels=dict(x="Colonne de la grille", y="Ligne de la grille", color="Forme Géométrique"),
    title="SOM : Projection des échantillons sur la grille (colorée par forme)"
)
fig.update_layout(
    width=800, height=700,
    template="plotly_dark",
    yaxis=dict(autorange="reversed")
)
fig.update_traces(marker=dict(size=3, opacity=0.6))
fig.show()

## 10. Génération de nouvelles images par sampling des neurones

Comme pour PCA, on génère de nouvelles images en sampling l'espace latent des neurones.
Ici, le latent est discret (indice du neurone), donc on sample des neurones existants ou proches.

Deux approches :
1. Afficher les neurones avec le plus d'activité (hits élevés) - ce sont les prototypes appris.
2. Interpoler entre deux neurones voisins pour obtenir une image "entre" deux prototypes.

In [16]:
bmu_indices = latent.array
hit_counts = np.zeros(model.n_neurons, dtype=int)
for idx in bmu_indices:
    hit_counts[idx] += 1

# Trouver les neurones avec le plus de hits
top_neuron_indices = np.argsort(hit_counts)[-6:][::-1]  # Top 6

fig = make_subplots(rows=2, cols=3, subplot_titles=[
    f"Neurone {idx}\n(hits: {hit_counts[idx]})" for idx in top_neuron_indices
])

for i, neuron_idx in enumerate(top_neuron_indices):
    row = i // 3 + 1
    col = i % 3 + 1
    
    neuron_img = model.W[neuron_idx].reshape(3, 32, 32).transpose(1, 2, 0)
    neuron_img = np.clip(neuron_img, 0.0, 1.0)
    
    fig.add_trace(
        go.Image(z=(neuron_img * 255).astype(np.uint8)),
        row=row, col=col
    )

fig.update_layout(
    height=500, width=1000,
    title_text="Les 6 neurones les plus actifs (prototypes importants)",
    template="plotly_dark"
)
for i in range(1, 7):
    fig.update_yaxes(showticklabels=False, row=(i-1)//3+1, col=(i-1)%3+1)
    fig.update_xaxes(showticklabels=False, row=(i-1)//3+1, col=(i-1)%3+1)
fig.show()

print(f"Hits des neurones sélectionnés : {[hit_counts[idx] for idx in top_neuron_indices]}")

Hits des neurones sélectionnés : [np.int64(203), np.int64(196), np.int64(184), np.int64(184), np.int64(169), np.int64(151)]
